# CSL7110 - Assignment 4 - Clustering and PageRank

**Student:** Akshay Kumar (M25DE1028)

**GitHub:** https://github.com/Akshaykumarky26/CSL7110_Assignment4

---

## Table of Contents

1. Environment and Setup
2. Part 1 - Clustering (Farthest-First + k-Means++) - 40 marks
   - 2.1 Complexity note
   - 2.2 Helper functions
   - 2.3 Experiment 1: k = 10, k1 = 50
   - 2.4 Experiment 2: k = 15, k1 = 100
   - 2.5 Experiment 3: k = 5, k1 = 200
3. Part 2 - Web-Search Inverted Index - 40 marks
   - 3.1 Classes, rules, assumption note
   - 3.2 Run actions.txt and validate against answers.txt
4. Part 3 - PageRank on Spark - 40 marks
   - 4.1 Algorithm
   - 4.2 small.txt (sanity check -> top ~ 0.036)
   - 4.3 whole.txt (top-5 and bottom-5)
   - 4.4 Sanity check: sum(PageRank) = 1.0


## 1. Environment and Setup

- Python 3.10
- PySpark 3.5.3 (local mode, `local[*]`)
- OpenJDK 11
- Datasets are placed alongside this notebook:
  - `spambase.data` (Part 1)
  - `actions.txt`, `answers.txt`, `webpages/` (Part 2)
  - `graphs/small.txt`, `graphs/whole.txt` (Part 3)

**How to run:** execute the cells top-to-bottom. The Spark context is created once at the top and reused across Parts 1 and 3.


In [1]:
import os, sys, time, math, random, platform
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

import pyspark
from pyspark import SparkContext, SparkConf
from pyspark.mllib.linalg import Vector, Vectors

print('Python  :', sys.version.split()[0])
print('PySpark :', pyspark.__version__)
print('Platform:', platform.platform())

conf = (SparkConf().setAppName('CSL7110_Assignment4')
        .setMaster('local[*]')
        .set('spark.driver.bindAddress', '127.0.0.1')
        .set('spark.ui.showConsoleProgress', 'false'))
sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel('ERROR')
print('Spark ready:', sc.version, '| master =', sc.master)

Python  : 3.13.13
PySpark : 4.1.1
Platform: macOS-26.4.1-arm64-arm-64bit-Mach-O


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 23:49:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready: 4.1.1 | master = local[*]


## 2. Part 1 - Clustering (40 Marks)

**Dataset:** `spambase.data` - 4601 points.
The original file has 57 real-valued feature attributes plus one `{0,1}` class label.
Since the assignment explicitly states *"4601 points with 58 dimensions"*, **all 58 numeric columns are retained as coordinates** - the class label is kept as an extra dimension rather than dropped. This matches the dimensionality stated in the assignment.

### 2.1 Complexity note (meets the O(|P| * k) requirement)

- `kcenter`  - each of the `k` rounds performs **one scan over |P|** to find the farthest point from the current centre set and then **one scan over |P|** to update the nearest-centre distance array. Total: `O(|P| * k)`.
- `kmeansPP` - each of the `k` rounds performs **one scan over |P|** to compute the D^2 sampling probabilities and **one scan over |P|** to update the nearest-centre distance array after the new centre is chosen. Total: `O(|P| * k)`.


In [2]:
# ---------- 2.2 Helper functions ----------
def readVectorsSeq(filename):
    """Read comma-separated feature points into a list of Vector."""
    vectors = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            feats = [float(x) for x in line.split(',')]
            vectors.append(Vectors.dense(feats))
    return vectors


def _sq_dist(a, b):
    """Squared L2 distance (pyspark-version-agnostic)."""
    try:
        return float(Vectors.sqdist(a, b))
    except AttributeError:
        aa = a.toArray() if hasattr(a, 'toArray') else a
        bb = b.toArray() if hasattr(b, 'toArray') else b
        diff = aa - bb
        return float((diff * diff).sum())


def kcenter(P, k):
    """Farthest-First Traversal. O(|P| * k)."""
    n = len(P)
    if k <= 0 or n == 0: return []
    centers = [P[0]]                                    # pick any point to start
    min_sq = [_sq_dist(P[i], centers[0]) for i in range(n)]
    for _ in range(1, k):
        far_idx, far_val = 0, -1.0
        for i in range(n):                              # scan of |P|
            if min_sq[i] > far_val:
                far_val, far_idx = min_sq[i], i
        c = P[far_idx]; centers.append(c)
        for i in range(n):                              # update nearest-centre distance
            d = _sq_dist(P[i], c)
            if d < min_sq[i]: min_sq[i] = d
    return centers


def kmeansPP(P, k):
    """k-Means++ seeding. O(|P| * k)."""
    n = len(P)
    if k <= 0 or n == 0: return []
    random.seed(42)                                     # reproducibility
    idx = random.randint(0, n - 1)
    centers = [P[idx]]
    min_sq = [_sq_dist(P[i], centers[0]) for i in range(n)]
    for _ in range(1, k):
        total = sum(min_sq)
        if total <= 0.0:
            idx = random.randint(0, n - 1)
        else:
            r = random.random() * total
            cum, idx = 0.0, n - 1
            for i in range(n):
                cum += min_sq[i]
                if cum >= r: idx = i; break
        c = P[idx]; centers.append(c)
        for i in range(n):
            d = _sq_dist(P[i], c)
            if d < min_sq[i]: min_sq[i] = d
    return centers


def kmeansObj(P, C):
    """Average squared distance of a point of P from its nearest centre."""
    if not P: return 0.0
    total = 0.0
    for p in P:
        best = float('inf')
        for c in C:
            d = _sq_dist(p, c)
            if d < best: best = d
        total += best
    return total / len(P)


# Load the dataset once; the three experiments reuse P.
P = readVectorsSeq('spambase.data')
print(f'|P| = {len(P)} points, dim = {len(P[0])}')

|P| = 4601 points, dim = 58


### 2.3 Experiment 1: k = 10, k1 = 50

Runs the three required steps: (1) `kcenter(P, k)` time; (2) `kmeansPP(P, k)` then `kmeansObj(P, C)`; (3) `kcenter(P, k1)` -> `kmeansPP(X, k)` -> `kmeansObj(P, C)`.


In [3]:
k, k1 = 10, 50
assert k < k1

t0 = time.time(); C_kc = kcenter(P, k);   t_kc = time.time() - t0
print(f'[kcenter ] k = {k:>2}: time = {t_kc:.4f} s, centers = {len(C_kc)}')

t0 = time.time(); C_pp = kmeansPP(P, k); t_pp = time.time() - t0
obj_pp = kmeansObj(P, C_pp)
print(f'[kmeansPP] k = {k:>2}: time = {t_pp:.4f} s, kmeansObj(direct)  = {obj_pp:.4f}')

t0 = time.time(); X = kcenter(P, k1);    t_k1 = time.time() - t0
C_cs = kmeansPP(X, k); obj_cs = kmeansObj(P, C_cs)
print(f'[coreset ] k1 = {k1}: kcenter time = {t_k1:.4f} s, |X| = {len(X)}')
print(f'           kmeansPP(X, {k}) -> kmeansObj(coreset) = {obj_cs:.4f}')

[kcenter ] k = 10: time = 0.0612 s, centers = 10
[kmeansPP] k = 10: time = 0.0587 s, kmeansObj(direct)  = 31251.6036
[coreset ] k1 = 50: kcenter time = 0.2920 s, |X| = 50
           kmeansPP(X, 10) -> kmeansObj(coreset) = 99261.9237


### 2.4 Experiment 2: k = 15, k1 = 100

In [4]:
k, k1 = 15, 100
assert k < k1

t0 = time.time(); C_kc = kcenter(P, k);   t_kc = time.time() - t0
print(f'[kcenter ] k = {k:>2}: time = {t_kc:.4f} s, centers = {len(C_kc)}')

t0 = time.time(); C_pp = kmeansPP(P, k); t_pp = time.time() - t0
obj_pp = kmeansObj(P, C_pp)
print(f'[kmeansPP] k = {k:>2}: time = {t_pp:.4f} s, kmeansObj(direct)  = {obj_pp:.4f}')

t0 = time.time(); X = kcenter(P, k1);    t_k1 = time.time() - t0
C_cs = kmeansPP(X, k); obj_cs = kmeansObj(P, C_cs)
print(f'[coreset ] k1 = {k1}: kcenter time = {t_k1:.4f} s, |X| = {len(X)}')
print(f'           kmeansPP(X, {k}) -> kmeansObj(coreset) = {obj_cs:.4f}')

[kcenter ] k = 15: time = 0.0858 s, centers = 15
[kmeansPP] k = 15: time = 0.0867 s, kmeansObj(direct)  = 23079.7012
[coreset ] k1 = 100: kcenter time = 0.5634 s, |X| = 100
           kmeansPP(X, 15) -> kmeansObj(coreset) = 163616.0197


### 2.5 Experiment 3: k = 5, k1 = 200

In [5]:
k, k1 = 5, 200
assert k < k1

t0 = time.time(); C_kc = kcenter(P, k);   t_kc = time.time() - t0
print(f'[kcenter ] k = {k:>2}: time = {t_kc:.4f} s, centers = {len(C_kc)}')

t0 = time.time(); C_pp = kmeansPP(P, k); t_pp = time.time() - t0
obj_pp = kmeansObj(P, C_pp)
print(f'[kmeansPP] k = {k:>2}: time = {t_pp:.4f} s, kmeansObj(direct)  = {obj_pp:.4f}')

t0 = time.time(); X = kcenter(P, k1);    t_k1 = time.time() - t0
C_cs = kmeansPP(X, k); obj_cs = kmeansObj(P, C_cs)
print(f'[coreset ] k1 = {k1}: kcenter time = {t_k1:.4f} s, |X| = {len(X)}')
print(f'           kmeansPP(X, {k}) -> kmeansObj(coreset) = {obj_cs:.4f}')

[kcenter ] k =  5: time = 0.0292 s, centers = 5
[kmeansPP] k =  5: time = 0.0284 s, kmeansObj(direct)  = 77727.0389
[coreset ] k1 = 200: kcenter time = 1.2289 s, |X| = 200
           kmeansPP(X, 5) -> kmeansObj(coreset) = 157413.2084


**Observations for Part 1:**
- `kcenter` running time scales roughly linearly with k (consistent with O(|P|*k)).
- The *direct* `kmeansPP(P, k)` objective is a good reference - as k grows the objective decreases (31k -> 23k going from k = 10 to k = 15).
- The coreset route (step 3) yields a larger `kmeansObj` on this dataset because `kcenter` deliberately selects outliers (farthest-first), so running `kmeansPP` on those outlier-only points inherits the extreme centres. This is a useful practical observation about the k1-coreset trade-off.


## 3. Part 2 - Web-Search Inverted Index (40 Marks)

### 3.1 Classes and rules

All eight classes are implemented: **MySet**, **Position**, **WordEntry**, **PageIndex**, **PageEntry**, **MyHashTable**, **InvertedPageIndex**, **SearchEngine**, along with the three actions `addPage`, `queryFindPagesWhichContainWord`, `queryFindPositionsOfWordInAPage`.

**API note:** For practical file loading, `PageEntry` and `SearchEngine` were parameterized with folder paths (`PageEntry(pageName, folder)`, `SearchEngine(folder)`). This does not change the required behaviour - just allows the class to read the actual file from disk.

**Processing rules:**
- lowercase every word;
- skip the 16 connector words `{a, an, the, they, these, this, for, is, are, was, of, or, and, does, will, whose}` **but still count them** for word-index purposes;
- replace punctuation `{} [] <> = () . , ; ' " ? # ! - :` with a space;
- normalise `(stack, stacks)`, `(structure, structures)`, `(application, applications)` to the singular form.


In [6]:
# ---------- 3.1 Class implementations ----------
CONNECTOR_WORDS = {
    'a', 'an', 'the', 'they', 'these', 'this', 'for', 'is',
    'are', 'was', 'of', 'or', 'and', 'does', 'will', 'whose',
}
PUNCTUATION = set(list("{}[]<>=().,;'\"?#!-:"))
PLURAL_TO_SINGULAR = {
    'stacks': 'stack',
    'structures': 'structure',
    'applications': 'application',
}

def normalize_word(w):
    w = w.lower()
    return PLURAL_TO_SINGULAR.get(w, w)

def tokenize(text):
    chars = [(' ' if ch in PUNCTUATION else ch) for ch in text]
    return [normalize_word(t) for t in ''.join(chars).split() if t.strip()]


class MySet:
    def __init__(self, iterable=None):
        self._data = set()
        if iterable is not None:
            for x in iterable: self._data.add(x)
    def addElement(self, element):     self._data.add(element)
    def union(self, other):            r = MySet(); r._data = self._data | other._data; return r
    def intersection(self, other):     r = MySet(); r._data = self._data & other._data; return r
    def __iter__(self):                return iter(self._data)
    def __len__(self):                 return len(self._data)
    def __contains__(self, x):         return x in self._data


class Position:
    def __init__(self, p, wordIndex):
        self._p, self._w = p, wordIndex
    def getPageEntry(self): return self._p
    def getWordIndex(self): return self._w


class WordEntry:
    def __init__(self, word):
        self._word, self._positions = word, []
    def addPosition(self, position):    self._positions.append(position)
    def addPositions(self, positions):  self._positions.extend(positions)
    def getAllPositionsForThisWord(self): return list(self._positions)
    def getWord(self): return self._word
    def getTermFrequency(self, page):
        count = sum(1 for pos in self._positions if pos.getPageEntry() is page)
        total = page.getTotalWords()
        return 0.0 if total == 0 else count / total


class PageIndex:
    def __init__(self):
        self._entries = {}
    def addPositionForWord(self, s, position):
        s = normalize_word(s)
        if s in self._entries:
            self._entries[s].addPosition(position)
        else:
            we = WordEntry(s); we.addPosition(position); self._entries[s] = we
    def getWordEntries(self): return list(self._entries.values())
    def getWordEntry(self, s): return self._entries.get(normalize_word(s))


class PageEntry:
    # API note: parameterized with folder for practical file loading.
    def __init__(self, pageName, folder):
        self._pageName = pageName
        self._pageIndex = PageIndex()
        with open(os.path.join(folder, pageName), 'r', encoding='utf-8') as f:
            text = f.read()
        tokens = tokenize(text)
        self._totalWords = len(tokens)
        # Word indices count connector words; connectors themselves are skipped.
        for idx, tok in enumerate(tokens, start=1):
            if tok in CONNECTOR_WORDS: continue
            self._pageIndex.addPositionForWord(tok, Position(self, idx))
    def getPageIndex(self): return self._pageIndex
    def getPageName(self): return self._pageName
    def getTotalWords(self): return self._totalWords


class MyHashTable:
    def __init__(self, size=1024):
        self._size = size
        self._buckets = {}
    def getHashIndex(self, s): return hash(normalize_word(s)) % self._size
    def addPositionsForWord(self, w):
        key = w.getWord()
        if key in self._buckets:
            self._buckets[key].addPositions(w.getAllPositionsForThisWord())
        else:
            new_we = WordEntry(key); new_we.addPositions(w.getAllPositionsForThisWord())
            self._buckets[key] = new_we
    def get(self, s):      return self._buckets.get(normalize_word(s))
    def contains(self, s): return normalize_word(s) in self._buckets


class InvertedPageIndex:
    def __init__(self):
        self._table = MyHashTable(); self._pages = {}
    def addPage(self, p):
        self._pages[p.getPageName()] = p
        for we in p.getPageIndex().getWordEntries():
            self._table.addPositionsForWord(we)
    def getPagesWhichContainWord(self, s):
        s = normalize_word(s); out = MySet()
        we = self._table.get(s)
        if we is None: return out
        for pos in we.getAllPositionsForThisWord():
            out.addElement(pos.getPageEntry())
        return out
    def getPage(self, name):  return self._pages.get(name)
    def hasPage(self, name):  return name in self._pages


class SearchEngine:
    # API note: parameterized with the webpages folder for practical I/O.
    def __init__(self, folder):
        self._invertedIndex = InvertedPageIndex()
        self._folder = folder
        self._outputs = []
    def _emit(self, s): print(s); self._outputs.append(s)
    def _addPage(self, pageName):
        try:
            p = PageEntry(pageName, self._folder)
        except FileNotFoundError:
            self._emit(f'No file {pageName} found'); return
        self._invertedIndex.addPage(p)
    def _queryFindPagesWhichContainWord(self, x):
        pages = self._invertedIndex.getPagesWhichContainWord(x)
        if len(pages) == 0:
            self._emit(f'No webpage contains word {x}')
        else:
            self._emit(', '.join(sorted(p.getPageName() for p in pages)))
    def _queryFindPositionsOfWordInAPage(self, x, y):
        if not self._invertedIndex.hasPage(y):
            self._emit(f'No webpage {y} found'); return
        page = self._invertedIndex.getPage(y)
        we = page.getPageIndex().getWordEntry(x)
        if we is None or not we.getAllPositionsForThisWord():
            self._emit(f'Webpage {y} does not contain word {x}'); return
        self._emit(', '.join(str(pos.getWordIndex())
                             for pos in we.getAllPositionsForThisWord()))
    def performAction(self, actionMessage):
        parts = actionMessage.strip().split()
        if not parts: return
        cmd = parts[0]
        if cmd == 'addPage' and len(parts) == 2:
            self._addPage(parts[1])
        elif cmd == 'queryFindPagesWhichContainWord' and len(parts) == 2:
            self._queryFindPagesWhichContainWord(parts[1])
        elif cmd == 'queryFindPositionsOfWordInAPage' and len(parts) == 3:
            self._queryFindPositionsOfWordInAPage(parts[1], parts[2])
        else:
            self._emit(f'Unknown action: {actionMessage}')

    def tfidfScore(self, word, pageName):
        page = self._invertedIndex.getPage(pageName)
        if page is None: return 0.0
        we = page.getPageIndex().getWordEntry(word)
        if we is None: return 0.0
        tf = we.getTermFrequency(page)
        N = len(self._invertedIndex._pages)
        n_w = len(self._invertedIndex.getPagesWhichContainWord(word))
        if n_w == 0 or N == 0: return 0.0
        return tf * math.log(N / n_w)

print('Part 2 classes defined.')

Part 2 classes defined.


### 3.2 Run actions and validate against answers.txt

The SearchEngine emits one line per action that produces output (there are 11 such queries out of the 17 actions). Each emitted line is compared against the corresponding line in `answers.txt`. An explicit **ALL PASSED** line is printed when every line matches.


In [7]:
se = SearchEngine('webpages')
with open('actions.txt', 'r', encoding='utf-8') as f:
    for ln in f:
        ln = ln.strip()
        if ln:
            se.performAction(ln)

with open('answers.txt', 'r', encoding='utf-8') as f:
    expected = [l.strip() for l in f if l.strip()]

print('\n--- Validation vs answers.txt ---')
all_ok = True
for i, (got, exp) in enumerate(zip(se._outputs, expected), 1):
    ok = got.strip() == exp.strip()
    all_ok &= ok
    marker = 'OK  ' if ok else 'DIFF'
    print(f'{marker} line {i:>2}: produced={got!r:<65}  expected={exp!r}')
print('\nALL PASSED' if all_ok else 'SOME FAILURES')

No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine

--- Validation vs answers.txt ---
OK   line  1: produced='No webpage contains word delhi'                                   expected='No webpage contains word delhi'
OK   line  2: produced='stack_datastructure_wiki'                                         expected='stack_datastructure_wiki'
OK   line  3: produced='stack_datastructure_wiki'                                         expected='stack_datastructure_wiki'
OK   line  4: produced='Webpage stack_datastructure_wiki does not contain word magazines'  expected='Webpage stack_datastructure_wiki does not contain word magazines'
OK   line  5: produced='No webpage contains word allain'                                  ex

## 4. Part 3 - PageRank on Spark (40 Marks)

### 4.1 Algorithm

$$r^{(i)} = \frac{1-\beta}{n}\,A + \beta \,M\, r^{(i-1)},\quad \beta = 0.8,\quad k = 40\text{ iterations}$$

Matrix $M$ is stored as an RDD of `(src, [dst])` adjacency lists. Duplicate directed edges between the same pair of nodes are collapsed via `.distinct()`. Dangling nodes (out-degree 0) are detected after `.distinct()`, and their leaked mass is redistributed uniformly across every node so the rank vector sums to exactly 1.0.

**Note on `small.txt` size:** the assignment text references 53 nodes, but the actual `small.txt` file in the reference repository (`pnijhara/PySpark-PageRank` at `graphs/small.txt`) contains **100 unique nodes and 950 distinct edges** after de-duplication. The top PageRank value **≈ 0.0357 ≈ 0.036** still matches the reference value given in the assignment, confirming the implementation is correct.


In [8]:
def build_graph(sc, path):
    raw = sc.textFile(path)
    edges = (raw.map(lambda s: s.strip())
                .filter(lambda s: s and not s.startswith('#'))
                .map(lambda s: s.split())
                .filter(lambda p: len(p) >= 2)
                .map(lambda p: (int(p[0]), int(p[1])))
                .distinct())
    nodes = edges.flatMap(lambda e: [e[0], e[1]]).distinct().collect()
    n = len(nodes)
    # (src, [dsts]); ensure every node has an entry so sinks get []
    links = edges.groupByKey().mapValues(lambda it: list(set(it)))
    all_nodes_pair = sc.parallelize([(x, []) for x in nodes])
    links = (links.union(all_nodes_pair)
                  .reduceByKey(lambda a, b: list(set(a) | set(b)))
                  .cache())
    return links, nodes, n


def _contrib(kv, beta):
    node, (neighbours, r) = kv
    if not neighbours: return []     # dangling handled separately
    share = beta * r / len(neighbours)
    return [(nb, share) for nb in neighbours]


def pagerank(sc, edges_path, beta=0.8, iters=40, trace=(1, 5, 10, 20, 40)):
    links, nodes, n = build_graph(sc, edges_path)
    uniq_edges = links.map(lambda kv: len(kv[1])).sum()
    print(f'[GRAPH] n = {n} nodes, unique edges = {uniq_edges}')

    ranks = links.mapValues(lambda _: 1.0 / n).cache()
    teleport = (1.0 - beta) / n
    trace_log = []
    for it in range(1, iters + 1):
        contribs = links.join(ranks).flatMap(lambda kv: _contrib(kv, beta))
        summed = contribs.reduceByKey(lambda a, b: a + b)

        dangling = (links.filter(lambda kv: len(kv[1]) == 0)
                         .map(lambda kv: (kv[0], None))
                         .leftOuterJoin(ranks)
                         .map(lambda kv: kv[1][1] or 0.0)
                         .sum())
        leaked = beta * dangling / n

        all_nodes = links.mapValues(lambda _: 0.0)
        ranks = (all_nodes.leftOuterJoin(summed)
                          .mapValues(lambda v: (v[1] or 0.0) + teleport + leaked)
                          .cache())

        if it in trace:
            top = ranks.takeOrdered(1, key=lambda kv: -kv[1])[0]
            total = ranks.values().sum()
            line = (f'[iter {it:2d}] top = (node {top[0]}, r = {top[1]:.6f}), '
                    f'sum(r) = {total:.6f}')
            print(line); trace_log.append((it, top, total))
    return ranks, trace_log


def top_bottom(ranks, k=5):
    top_k = ranks.takeOrdered(k, key=lambda kv: -kv[1])
    bot_k = ranks.takeOrdered(k, key=lambda kv: kv[1])
    return top_k, bot_k

print('PageRank helpers defined.')

PageRank helpers defined.


### 4.2 Sanity run on `small.txt` (reference top score ≈ 0.036)

In [9]:
print('=== small.txt ===')
ranks_small, trace_s = pagerank(sc, 'graphs/small.txt', beta=0.8, iters=40)
top5_s, bot5_s = top_bottom(ranks_small, 5)
print('\nTop 5 (highest PageRank):')
for n, r in top5_s: print(f'  node {n:>4}: {r:.6f}')
print('Bottom 5 (lowest PageRank):')
for n, r in bot5_s: print(f'  node {n:>4}: {r:.6f}')

=== small.txt ===
[GRAPH] n = 100 nodes, unique edges = 950
[iter  1] top = (node 14, r = 0.035585), sum(r) = 1.000000
[iter  5] top = (node 53, r = 0.035722), sum(r) = 1.000000
[iter 10] top = (node 53, r = 0.035731), sum(r) = 1.000000
[iter 20] top = (node 53, r = 0.035731), sum(r) = 1.000000
[iter 40] top = (node 53, r = 0.035731), sum(r) = 1.000000

Top 5 (highest PageRank):
  node   53: 0.035731
  node   14: 0.034171
  node   40: 0.033630
  node    1: 0.030006
  node   27: 0.029720
Bottom 5 (lowest PageRank):
  node   85: 0.003410
  node   59: 0.003670
  node   81: 0.003695
  node   37: 0.003808
  node   89: 0.003922


### 4.3 Full run on `whole.txt` (1000 nodes)

In [10]:
print('=== whole.txt ===')
ranks_whole, trace_w = pagerank(sc, 'graphs/whole.txt', beta=0.8, iters=40)
top5_w, bot5_w = top_bottom(ranks_whole, 5)
print('\nTop 5 (highest PageRank):')
for n, r in top5_w: print(f'  node {n:>4}: {r:.6f}')
print('Bottom 5 (lowest PageRank):')
for n, r in bot5_w: print(f'  node {n:>4}: {r:.6f}')

=== whole.txt ===
[GRAPH] n = 1000 nodes, unique edges = 8161
[iter  1] top = (node 263, r = 0.001876), sum(r) = 1.000000
[iter  5] top = (node 263, r = 0.002021), sum(r) = 1.000000
[iter 10] top = (node 263, r = 0.002020), sum(r) = 1.000000
[iter 20] top = (node 263, r = 0.002020), sum(r) = 1.000000
[iter 40] top = (node 263, r = 0.002020), sum(r) = 1.000000

Top 5 (highest PageRank):
  node  263: 0.002020
  node  537: 0.001943
  node  965: 0.001925
  node  243: 0.001853
  node  285: 0.001827
Bottom 5 (lowest PageRank):
  node  558: 0.000329
  node   93: 0.000351
  node   62: 0.000353
  node  424: 0.000355
  node  408: 0.000388


### 4.4 Sanity check - the rank vector must sum to 1.0

Because dangling-node leaked mass is redistributed uniformly to every node, the sum of PageRank values across all nodes is exactly 1.0 at every iteration. The cell below verifies this on both graphs.


In [11]:
for label, ranks in [('small.txt', ranks_small), ('whole.txt', ranks_whole)]:
    s = ranks.values().sum()
    print(f'sum(PageRank) on {label:>10} after 40 iterations = {s:.10f}')

sum(PageRank) on  small.txt after 40 iterations = 1.0000000000
sum(PageRank) on  whole.txt after 40 iterations = 1.0000000000


**Observations for Part 3:**
- On `small.txt` the top score stabilises at ≈ 0.035731 (≈ 0.036 as required).
- On `whole.txt` the top-5 nodes are 263, 537, 965, 243, 285; the bottom-5 are 558, 93, 62, 424, 408.
- PageRank converges extremely fast - the top-1 value is already stable by iteration 10; 40 iterations are kept for the assignment requirement.
- The sanity-check cell confirms `sum(r) = 1.0` to ten decimal places.


In [12]:
sc.stop()
print('Spark context stopped.')

Spark context stopped.


---
## Summary

- **Part 1** - `kcenter` and `kmeansPP` both in O(|P| * k); three experiments run (k = 10/k1 = 50, k = 15/k1 = 100, k = 5/k1 = 200) with printed timings and objective values.
- **Part 2** - Inverted-index + SearchEngine; validated 11/11 matches against `answers.txt`, printed `ALL PASSED`.
- **Part 3** - 40-iteration PageRank with β = 0.8. Top score on `small.txt` ≈ 0.0357 (matches 0.036 reference). Top-5/bottom-5 reported on `whole.txt`. Sum-to-1 sanity check passes.

